# NER propre à Djibouti — Baselines (Mako)

Master 2 IAMD, Université de Djibouti — module TALN.

**Question :** peut-on extraire les institutions, lieux, personnes et dates dans des textes français de Djibouti ?

**Plan du notebook**
1. Charger les données et les annotations
2. Baseline 1 : spaCy `fr_core_news_sm` seul  ← *section en cours*
3. Baseline 2 : spaCy + `EntityRuler` (lexique local + dates)
4. Comparaison des deux baselines (sur le test)
5. Analyse de 10 erreurs
6. Démo avec `displacy`

## Section 1 — Charger les données

**Ce qu'on fait :**
- on lit les deux fichiers : les textes (`corpus_ner.csv`) et les annotations (`annotations.csv`) ;
- pour chaque segment, on regroupe ses entités sous forme de triplets `(debut, fin, type)` ;
- on sépare le **dev** (pour écrire les règles) et le **test** (pour mesurer, sans y toucher).

**Pourquoi des triplets ?** C'est le format attendu par `src/evaluate.py`. Exemple : dans `"Il est à Djibouti."`, l'entité `Djibouti` donne le triplet `(10, 18, "LOC")`.

In [1]:
from pathlib import Path
from collections import Counter

import pandas as pd

# Le notebook est dans notebooks/ : les données sont un dossier plus haut.
RACINE = Path("..").resolve()
DATA = RACINE / "data"

corpus = pd.read_csv(DATA / "corpus_ner.csv", encoding="utf-8")
annotations = pd.read_csv(DATA / "annotations.csv", encoding="utf-8", keep_default_na=False)

print("Segments dans le corpus :", len(corpus))
print("Lignes d'annotation     :", len(annotations))
print("Colonnes du corpus      :", list(corpus.columns))
print("Colonnes des annotations:", list(annotations.columns))

Segments dans le corpus : 126
Lignes d'annotation     : 84
Colonnes du corpus      : ['id', 'titre', 'rubrique', 'url', 'date', 'fr', 'split']
Colonnes des annotations: ['id_segment', 'split', 'texte', 'entite', 'type', 'debut', 'fin', 'a_verifier', 'remarque']


**Attention : segments sans entité.** Le fichier d'annotations n'a aucune ligne pour un segment sans entité. Mais ces segments comptent dans l'évaluation : si le modèle y invente une entité, c'est une erreur (fausse). On leur donne donc une liste **vide**.

In [2]:
# gold[id_segment] = liste de triplets (debut, fin, type)
gold = {id_seg: [] for id_seg in corpus["id"]}

for ligne in annotations.itertuples():
    gold[ligne.id_segment].append((int(ligne.debut), int(ligne.fin), ligne.type))

# On trie pour avoir un ordre stable
for id_seg in gold:
    gold[id_seg].sort()

# Vérification : tous les segments annotés existent dans le corpus ?
inconnus = set(annotations["id_segment"]) - set(corpus["id"])
print("Segments annotés absents du corpus :", len(inconnus))

n_vides = sum(1 for ents in gold.values() if not ents)
print("Segments avec au moins une entité  :", len(gold) - n_vides)
print("Segments sans entité (liste vide)  :", n_vides)

Segments annotés absents du corpus : 0
Segments avec au moins une entité  : 48
Segments sans entité (liste vide)  : 78


**Dev et test.** On crée, pour chaque split, trois listes de même longueur et dans le même ordre : les ids, les textes, et les entités de la vérité terrain.

In [3]:
def charger_split(nom):
    sous = corpus[corpus["split"] == nom]
    ids = list(sous["id"])
    textes = list(sous["fr"])
    verite = [gold[i] for i in ids]
    return ids, textes, verite

ids_dev, textes_dev, gold_dev = charger_split("dev")
ids_test, textes_test, gold_test = charger_split("test")

print("dev  :", len(ids_dev), "segments")
print("test :", len(ids_test), "segments")

dev  : 84 segments
test : 42 segments


**Comptes réels par split et par type.**

In [4]:
TYPES = ["ORG", "LOC", "PER", "DATE", "FORMATION"]

def compter(gold_liste):
    c = Counter(t for ents in gold_liste for (_, _, t) in ents)
    return {t: c.get(t, 0) for t in TYPES}

tableau = pd.DataFrame({"dev": compter(gold_dev), "test": compter(gold_test)})
tableau["total"] = tableau["dev"] + tableau["test"]
tableau.loc["TOTAL"] = tableau.sum()
tableau

,dev,test,total
ORG,17,13,30
LOC,16,4,20
PER,14,7,21
DATE,10,3,13
FORMATION,0,0,0
TOTAL,57,27,84


**Lecture :**
- le **test est petit** : peu d'entités par type, donc un seul changement fait bouger le score de plusieurs points ;
- **FORMATION = 0** : aucun diplôme dans ce corpus, on n'en a pas inventé.

In [5]:
# Un exemple concret : un segment, ses triplets, et le texte coupé avec ces positions
exemple = "A03_S002"
texte = corpus.loc[corpus["id"] == exemple, "fr"].iloc[0]
print(texte, "\n")
for debut, fin, type_ in gold[exemple]:
    print(f"({debut}, {fin}, {type_!r:6})  ->  {texte[debut:fin]!r}")

Réunis en Assemblée générale extraordinaire au siège de la Fédération, ils ont voté à l'unanimité la révocation du Comité directeur et de son président, Souleiman Hassan Waberi. 

(115, 131, 'ORG' )  ->  'Comité directeur'
(153, 176, 'PER' )  ->  'Souleiman Hassan Waberi'


In [6]:
# Contrôle final : pour TOUTES les annotations, le texte coupé doit être l'entité notée
ok = all(
    corpus.loc[corpus["id"] == l.id_segment, "fr"].iloc[0][l.debut:l.fin] == l.entite
    for l in annotations.itertuples()
)
print("Toutes les positions correspondent au texte :", ok)

Toutes les positions correspondent au texte :

 True


**Fin de la Section 1.** On a `gold_dev`, `gold_test`, `textes_dev`, `textes_test`, `ids_dev`, `ids_test`. La Section 2 les utilisera pour la baseline spaCy.

## Section 2 — Baseline 1 : spaCy seul

**Ce qu'on fait :** on passe nos textes dans le modèle `fr_core_news_sm` **sans aucune règle ajoutée**, puis on compare ce qu'il trouve avec notre vérité terrain.

**Le modèle, en bref :** c'est un petit modèle français de spaCy, déjà entraîné par ses auteurs. Il ne connaît pas Djibouti en particulier. Cellule suivante : sa fiche d'identité (nom, version, langue, étiquettes).

In [7]:
import sys
import spacy

nlp = spacy.load("fr_core_news_sm")
meta = nlp.meta

print("spaCy (bibliothèque) :", spacy.__version__)
print("Modèle               :", meta["lang"] + "_" + meta["name"])
print("Version du modèle    :", meta["version"])
print("Langue               :", meta["lang"])
print("Composants           :", nlp.pipe_names)
print("Étiquettes du NER    :", nlp.get_pipe("ner").labels)
print("Données d'entraînement:", [s["name"] for s in meta["sources"]])

spaCy (bibliothèque) : 3.8.18
Modèle               : fr_core_news_sm
Version du modèle    : 3.8.0
Langue               : fr
Composants           : ['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Étiquettes du NER    : ('LOC', 'MISC', 'ORG', 'PER')
Données d'entraînement: ['UD French Sequoia v2.8', 'WikiNER', 'spaCy lookups data']


**À retenir (limites) :**
- le modèle ne prédit que **4 étiquettes** : `PER`, `ORG`, `LOC`, `MISC`. Il ne connaît ni `DATE` ni `FORMATION` ;
- c'est le plus petit modèle (`sm`) : rapide mais moins précis ;
- il a été entraîné sur des textes généraux (articles Wikipédia, corpus Sequoia), pas sur la presse de Djibouti.

**On importe `src/evaluate.py`** (le calcul précision / rappel / F1) :

In [8]:
sys.path.insert(0, str(RACINE / "src"))
from evaluate import evaluer, afficher, erreurs

print("Fonctions importées :", evaluer.__name__, afficher.__name__, erreurs.__name__)

Fonctions importées : evaluer afficher erreurs


**Prédire.** La fonction `predire` fait trois choses :
1. elle passe chaque texte dans spaCy ;
2. elle lit pour chaque entité `start_char` et `end_char` : ce sont des positions **en caractères**, comme dans nos annotations ;
3. elle garde `PER`, `ORG`, `LOC` et **ignore `MISC`**, car nous n'avons pas ce type dans notre guide (MISC veut dire « divers » : spaCy y met tout ce qui n'entre pas ailleurs).

Elle compte aussi ce qu'elle ignore, pour qu'on voie ce qu'on perd.

In [9]:
GARDES = ["PER", "ORG", "LOC"]

def predire(textes):
    """Retourne (prédictions, compteur des étiquettes ignorées).
    prédictions = une liste par texte de triplets (debut, fin, type)."""
    resultat, ignorees = [], Counter()
    for doc in nlp.pipe(textes):
        ents = []
        for ent in doc.ents:
            if ent.label_ in GARDES:
                ents.append((ent.start_char, ent.end_char, ent.label_))
            else:
                ignorees[ent.label_] += 1
        resultat.append(ents)
    return resultat, ignorees

pred_dev, ignorees_dev = predire(textes_dev)
pred_test, ignorees_test = predire(textes_test)

def total(pred):
    return Counter(t for ents in pred for (_, _, t) in ents)

print("TEST — entités gardées     :", dict(total(pred_test)), "| ignorées :", dict(ignorees_test))
print("DEV  — entités gardées     :", dict(total(pred_dev)),  "| ignorées :", dict(ignorees_dev))

TEST — entités gardées     : {'ORG': 19, 'LOC': 7, 'PER': 4} | ignorées : {'MISC': 2}
DEV  — entités gardées     : {'ORG': 19, 'PER': 21, 'LOC': 29} | ignorées : {'MISC': 4}


**Vérité terrain filtrée.** Le score principal porte sur `PER`, `ORG`, `LOC` seulement, car ce sont les seuls types que spaCy peut prédire. On retire donc `DATE` (et `FORMATION`) de la vérité terrain **pour ce calcul**. Sinon on punirait spaCy pour des dates qu'il ne sait pas prédire.

In [10]:
def filtrer(gold_liste, types):
    return [[e for e in ents if e[2] in types] for ents in gold_liste]

gold_test_3 = filtrer(gold_test, GARDES)
gold_dev_3 = filtrer(gold_dev, GARDES)

print("Entités de la vérité terrain gardées (PER, ORG, LOC) :")
print("  test :", sum(len(e) for e in gold_test_3), "sur", sum(len(e) for e in gold_test))
print("  dev  :", sum(len(e) for e in gold_dev_3), "sur", sum(len(e) for e in gold_dev))

Entités de la vérité terrain gardées (PER, ORG, LOC) :
  test : 24 sur 27
  dev  : 47 sur 57


### Résultat sur le TEST (score principal)

Une entité est **juste** seulement si début, fin **et** type sont exacts. `TOUT` = PER + ORG + LOC.

In [11]:
res_test = evaluer(gold_test_3, pred_test)
afficher(res_test)

Type         Précision    Rappel      F1  Support
LOC               0.29      0.50    0.36        4
ORG               0.47      0.69    0.56       13
PER               0.75      0.43    0.55        7
TOUT              0.47      0.58    0.52       24


**Comment lire :**
- **Précision** : parmi ce que spaCy a trouvé, quelle part est juste ;
- **Rappel** : parmi ce qu'il fallait trouver, quelle part spaCy a trouvée ;
- **Support** : nombre d'entités réelles du type dans le test ;
- le test est **petit** : un seul changement fait bouger un score de plusieurs points.

### DATE : non prédit par spaCy

spaCy `fr_core_news_sm` n'a pas d'étiquette `DATE`. Il ne peut donc **jamais** en prédire. On le montre pour le rapport, mais on ne le mélange pas au score principal.

In [12]:
def date_seule(gold_liste):
    return filtrer(gold_liste, ["DATE"])

for nom, gold_x in [("test", gold_test), ("dev", gold_dev)]:
    gd = date_seule(gold_x)
    vide = [[] for _ in gd]                     # spaCy ne prédit aucune DATE
    r = evaluer(gd, vide)["DATE"]
    print(f"DATE ({nom}) : support = {r[3]}, prédites par spaCy = 0, rappel = {r[1]:.2f}  -> non prédit par spaCy")
    print("  (précision non définie : 0 prédiction)")

DATE (test) : support = 3, prédites par spaCy = 0, rappel = 0.00  -> non prédit par spaCy
  (précision non définie : 0 prédiction)
DATE (dev) : support = 10, prédites par spaCy = 0, rappel = 0.00  -> non prédit par spaCy
  (précision non définie : 0 prédiction)


### Mêmes chiffres sur le DEV (à titre indicatif)

On ne s'en sert **pas** pour décider quoi que ce soit : le dev sert plus tard à écrire les règles (Section 3). Il est seulement là pour voir si les chiffres du test sont d'un ordre de grandeur comparable.

In [13]:
res_dev = evaluer(gold_dev_3, pred_dev)
afficher(res_dev)

Type         Précision    Rappel      F1  Support
LOC               0.41      0.75    0.53       16
ORG               0.58      0.65    0.61       17
PER               0.52      0.79    0.63       14
TOUT              0.49      0.72    0.59       47


### Combien d'entités MISC sont ignorées ?

Ce sont des entités que spaCy a trouvées mais qu'on ne compare pas (pas de type `MISC` dans notre guide). Voici lesquelles, pour voir si on perd quelque chose d'important.

In [14]:
print("MISC ignorées : test =", ignorees_test.get("MISC", 0), "| dev =", ignorees_dev.get("MISC", 0))
print("Autres étiquettes ignorées :", {k: v for k, v in (ignorees_test + ignorees_dev).items() if k != "MISC"} or "aucune")

print("\nExemples de MISC (test) :")
for doc, id_ in zip(nlp.pipe(textes_test), ids_test):
    for ent in doc.ents:
        if ent.label_ == "MISC":
            print(f"  {id_}  {ent.text!r}")

MISC ignorées : test = 2 | dev = 4
Autres étiquettes ignorées : aucune

Exemples de MISC (test) :


  A03_S007  'Me Abeyazide'
  A11_S001  'Formation professionnelle'


### À quoi ressemblent les erreurs ? 5 segments du test

**Choix des segments :** les 5 **premiers** segments du test (dans l'ordre du fichier) qui contiennent au moins une erreur sur `PER/ORG/LOC`. Ce n'est donc **pas** un échantillon représentatif : on montre exprès des segments avec erreurs.

- *manquée* = dans la vérité terrain, mais spaCy ne l'a pas trouvée avec les mêmes frontières ET le même type ;
- *fausse* = spaCy l'a prédite, mais elle n'est pas dans la vérité terrain.

In [15]:
def montrer(id_, texte, g, p):
    print(f"[{id_}] {texte}")
    print("  spaCy  :", [(texte[d:f], t) for d, f, t in p] or "aucune")
    print("  vérité :", [(texte[d:f], t) for d, f, t in g] or "aucune")
    for genre, ent, mot in erreurs(texte, g, p):
        print(f"  -> {genre:8} {mot!r} ({ent[2]})")
    print()

compte = 0
for id_, texte, g, p in zip(ids_test, textes_test, gold_test_3, pred_test):
    if erreurs(texte, g, p):
        montrer(id_, texte, g, p)
        compte += 1
        if compte == 5:
            break

[A03_S002] Réunis en Assemblée générale extraordinaire au siège de la Fédération, ils ont voté à l'unanimité la révocation du Comité directeur et de son président, Souleiman Hassan Waberi.
  spaCy  : [('Réunis', 'LOC'), ('Assemblée générale', 'ORG'), ('Fédération', 'LOC'), ('Comité directeur', 'ORG'), ('Souleiman Hassan Waberi', 'PER')]
  vérité : [('Comité directeur', 'ORG'), ('Souleiman Hassan Waberi', 'PER')]
  -> fausse   'Réunis' (LOC)
  -> fausse   'Assemblée générale' (ORG)
  -> fausse   'Fédération' (LOC)

[A03_S004] L'Assemblée générale extraordinaire avait été convoquée à la demande des clubs membres afin d'examiner la situation de la Fédération et son avenir institutionnel.
  spaCy  : [('Assemblée générale', 'ORG'), ('Fédération', 'ORG')]
  vérité : aucune
  -> fausse   'Assemblée générale' (ORG)
  -> fausse   'Fédération' (ORG)

[A03_S007] La réunion s'est tenue dans un cadre formel, sous le contrôle du consultant Me Abeyazide et de l'huissier de justice Habib Alwan.
  spaC

**Fin de la Section 2.** On a `pred_test`, `pred_dev` et les scores de la baseline 1 (`res_test`, `res_dev`). La Section 3 ajoutera des règles locales (`EntityRuler`) en regardant **seulement le dev**.